# Descenso del gradiente para regresión lineal (una variable)

**Modelo:** $f_{w,b}(x)=wx+b$

**Costo:** $J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2$

**Objetivo:** encontrar $w$ y $b$ que minimicen $J$ mediante actualizaciones iterativas:

$$w := w-\alpha\frac{\partial J}{\partial w},\qquad b := b-\alpha\frac{\partial J}{\partial b}$$

(actualización **simultánea** de ambos parámetros; $\alpha$ es la tasa de aprendizaje).

## Paso 1. Librerías y datos

Usamos un conjunto pequeño: tamaño de casa en 1000 sqft ($x$) y precio en miles de dólares ($y$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_train = np.array([1.0, 1.5, 2.0, 2.5, 3.0])
y_train = np.array([300.0, 380.0, 500.0, 560.0, 700.0])
m = x_train.shape[0]
print(f"m = {m} ejemplos")

plt.scatter(x_train, y_train, marker='x', c='r')
plt.xlabel('Tamaño (1000 sqft)'); plt.ylabel('Precio (miles de $)')
plt.title('Datos de entrenamiento'); plt.show()

## Paso 2. Función de costo

Mide el error cuadrático medio (dividido entre 2 para simplificar la derivada):

$$J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}(wx^{(i)}+b-y^{(i)})^2$$

In [ ]:
def compute_cost(x, y, w, b):
    m = x.shape[0]
    cost = 0.0
    for i in range(m):
        f_wb = w * x[i] + b          # predicción del ejemplo i
        cost += (f_wb - y[i]) ** 2   # error al cuadrado
    return cost / (2 * m)

print("J(0,0)     =", compute_cost(x_train, y_train, 0, 0))
print("J(200,100) =", compute_cost(x_train, y_train, 200, 100))

## Paso 3. El gradiente

Derivando $J$ (la regla de la cadena genera un factor 2 que cancela el $\frac12$):

$$\frac{\partial J}{\partial w}=\frac1m\sum_{i=1}^{m}(f_{w,b}(x^{(i)})-y^{(i)})\,x^{(i)},\qquad
\frac{\partial J}{\partial b}=\frac1m\sum_{i=1}^{m}(f_{w,b}(x^{(i)})-y^{(i)})$$

In [ ]:
def compute_gradient(x, y, w, b):
    m = x.shape[0]
    dj_dw, dj_db = 0.0, 0.0
    for i in range(m):
        err = (w * x[i] + b) - y[i]   # f_wb - y
        dj_dw += err * x[i]
        dj_db += err
    return dj_dw / m, dj_db / m

# Verificación numérica del gradiente (diferencias finitas)
w0, b0, eps = 50.0, 20.0, 1e-5
num_dw = (compute_cost(x_train, y_train, w0+eps, b0) - compute_cost(x_train, y_train, w0-eps, b0)) / (2*eps)
num_db = (compute_cost(x_train, y_train, w0, b0+eps) - compute_cost(x_train, y_train, w0, b0-eps)) / (2*eps)
print("Analítico:", compute_gradient(x_train, y_train, w0, b0))
print("Numérico :", (num_dw, num_db))

## Paso 4. Algoritmo de descenso del gradiente

En cada iteración: calcular el gradiente, actualizar $w$ y $b$ **simultáneamente** y guardar $J$ para graficarlo después.

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters):
    w, b = w_in, b_in
    J_history = [compute_cost(x, y, w, b)]
    p_history = [(w, b)]
    for it in range(num_iters):
        dj_dw, dj_db = compute_gradient(x, y, w, b)
        w = w - alpha * dj_dw      # ambos gradientes ya calculados con (w,b) anteriores
        b = b - alpha * dj_db
        J_history.append(compute_cost(x, y, w, b))
        p_history.append((w, b))
        if it % max(1, num_iters // 10) == 0:
            print(f"Iter {it:5d}: J = {J_history[-1]:10.4f}  w = {w:8.4f}  b = {b:8.4f}")
    return w, b, J_history, p_history

## Paso 5. Ejecutar

Inicializamos $w=b=0$ con $\alpha=0.1$ y 1000 iteraciones.

In [ ]:
alpha, iters = 0.1, 1000
w_final, b_final, J_hist, p_hist = gradient_descent(x_train, y_train, 0.0, 0.0, alpha, iters)
print(f"\nResultado: w = {w_final:.4f}, b = {b_final:.4f}, J = {J_hist[-1]:.4f}")

## Paso 6. ¿Cómo se reduce el costo en cada iteración?

Si el descenso funciona, $J$ **disminuye en cada iteración**. A la derecha se usa escala logarítmica en $y$ para ver la convergencia.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(J_hist[:100]); ax1.set_title('Costo vs iteración (primeras 100)')
ax2.plot(J_hist); ax2.set_yscale('log'); ax2.set_title('Costo vs iteración (escala log)')
for ax in (ax1, ax2):
    ax.set_xlabel('Iteración'); ax.set_ylabel('J(w,b)')
plt.tight_layout(); plt.show()

print("¿J decrece monótonamente?", all(np.diff(J_hist) <= 1e-12))

## Paso 7. Ajuste final y trayectoria sobre el contorno de $J$

Izquierda: la recta aprendida. Derecha: el camino $(w,b)$ recorrido sobre las curvas de nivel de $J$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.scatter(x_train, y_train, marker='x', c='r', label='Datos')
ax1.plot(x_train, w_final * x_train + b_final, c='b', label=f'f(x)={w_final:.1f}x+{b_final:.1f}')
ax1.set_xlabel('Tamaño (1000 sqft)'); ax1.set_ylabel('Precio (miles de $)'); ax1.legend()

W, B = np.meshgrid(np.linspace(-50, 300, 120), np.linspace(-100, 300, 120))
Z = np.array([[compute_cost(x_train, y_train, w, b) for w in W[0]] for b in B[:, 0]])
ax2.contour(W, B, Z, levels=np.logspace(2, 5.3, 20), cmap='viridis')
p = np.array(p_hist)
ax2.plot(p[:60, 0], p[:60, 1], 'r.-', label='Trayectoria (60 primeras)')
ax2.plot(p[-1, 0], p[-1, 1], 'k*', ms=12, label='Final')
ax2.set_xlabel('w'); ax2.set_ylabel('b'); ax2.legend(); ax2.set_title('Contorno de J(w,b)')
plt.tight_layout(); plt.show()

## Paso 8. Efecto de la tasa de aprendizaje $\alpha$

- $\alpha$ muy pequeña: convergencia lenta.
- $\alpha$ adecuada: convergencia rápida.
- $\alpha$ muy grande: el costo **diverge**.

In [ ]:
import io, contextlib
plt.figure(figsize=(7, 4))
for a in [0.001, 0.01, 0.1, 0.9]:
    with contextlib.redirect_stdout(io.StringIO()):
        *_, Jh, _ = gradient_descent(x_train, y_train, 0.0, 0.0, a, 50)
    plt.plot(Jh, label=f'α={a}')
plt.yscale('log'); plt.xlabel('Iteración'); plt.ylabel('J(w,b)'); plt.legend()
plt.title('Costo según α (50 iteraciones)'); plt.show()

## Paso 9. Predicción

Con los parámetros aprendidos predecimos el precio de casas nuevas.

In [ ]:
for size in [1.2, 2.0, 3.5]:
    print(f"Casa de {size*1000:.0f} sqft -> ${(w_final*size + b_final)*1000:,.0f}")

## Conclusiones

1. Definimos $f_{w,b}$ y $J(w,b)$.
2. Calculamos $\partial J/\partial w$ y $\partial J/\partial b$ (y los verificamos numéricamente).
3. Actualizamos $w,b$ simultáneamente restando $\alpha\cdot$gradiente.
4. Graficamos $J$ por iteración para comprobar la convergencia.

**Ejercicio:** cambia los valores iniciales de $w,b$, el número de iteraciones y $\alpha$, y observa el efecto.